# 07 — Calibration & Threshold Optimisation

Calibrate the top-3 models (RF, XGBoost, LogReg) with Platt scaling and isotonic regression, then find the cost-optimal decision threshold under C(FN)=5, C(FP)=1.

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import (brier_score_loss, f1_score,
                             precision_recall_curve, roc_auc_score)
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
C_FN, C_FP = 5, 1

MODELS_DIR  = os.path.abspath('../models')
FIGURES_DIR = os.path.abspath('../figures')
DATA_DIR    = os.path.abspath('../data/processed')
for d in [MODELS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)
print("Paths OK")


In [ ]:
# ── Load data ────────────────────────────────────────────────
def load_splits():
    """Load pre-computed train/val/test splits or fall back to synthetic data."""
    parquet = os.path.join(DATA_DIR, 'modeling_table.parquet')
    if os.path.exists(parquet):
        df = pd.read_parquet(parquet)
        from sklearn.model_selection import train_test_split
        feature_cols = [c for c in df.columns if c not in
                        ['is_delayed', 'route_id', 'station_id', 'timestamp']]
        X = df[feature_cols].values.astype(float)
        y = df['is_delayed'].values
        X_tv, X_test, y_tv, y_test = train_test_split(
            X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y)
        X_train, X_val, y_train, y_val = train_test_split(
            X_tv, y_tv, test_size=0.176, random_state=RANDOM_STATE, stratify=y_tv)
        return X_train, X_val, X_test, y_train, y_val, y_test, feature_cols
    else:
        print("⚠  modeling_table.parquet not found — using synthetic data.")
        from sklearn.datasets import make_classification
        X, y = make_classification(n_samples=12000, n_features=20,
                                   n_informative=12, weights=[0.65, 0.35],
                                   random_state=RANDOM_STATE)
        feature_cols = [f'feat_{i}' for i in range(20)]
        from sklearn.model_selection import train_test_split
        X_tv, X_test, y_tv, y_test = train_test_split(
            X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y)
        X_train, X_val, y_train, y_val = train_test_split(
            X_tv, y_tv, test_size=0.176, random_state=RANDOM_STATE, stratify=y_tv)
        return X_train, X_val, X_test, y_train, y_val, y_test, feature_cols

X_train, X_val, X_test, y_train, y_val, y_test, FEATURE_COLS = load_splits()
print(f"Train {X_train.shape}, Val {X_val.shape}, Test {X_test.shape}")


In [ ]:
# ── Load top-3 models ─────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

MODEL_FILES = {
    'RandomForest': 'rf.joblib',
    'XGBoost':      'xgb.joblib',
    'LogReg':       'logreg.joblib',
}

def load_or_train(name, path):
    if os.path.exists(path):
        return joblib.load(path)
    print(f"  ⚠  {name} not found — training fallback model.")
    if name == 'RandomForest':
        m = RandomForestClassifier(n_estimators=200, max_depth=8,
                                   random_state=RANDOM_STATE, n_jobs=-1)
    elif name == 'XGBoost':
        m = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8, use_label_encoder=False,
                               eval_metric='logloss', random_state=RANDOM_STATE)
    else:
        m = LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE)
    m.fit(X_train, y_train)
    joblib.dump(m, path)
    return m

models = {}
for name, fname in MODEL_FILES.items():
    path = os.path.join(MODELS_DIR, fname)
    models[name] = load_or_train(name, path)
    print(f"  Loaded: {name}")


In [ ]:
# ── Reliability curves & Brier scores (pre-calibration) ──────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Reliability Diagrams — Before Calibration', fontsize=14)

brier_before = {}
for ax, (name, model) in zip(axes, models.items()):
    proba = model.predict_proba(X_val)[:, 1]
    frac_pos, mean_pred = calibration_curve(y_val, proba, n_bins=10, strategy='uniform')
    bs = brier_score_loss(y_val, proba)
    brier_before[name] = bs
    ax.plot(mean_pred, frac_pos, 's-', label=f'Model (Brier={bs:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect')
    ax.set_title(name)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'reliability_before.png'), dpi=150)
plt.show()
print("Brier scores (before calibration):", brier_before)


In [ ]:
# ── Apply calibration: Platt scaling & isotonic regression ───
from sklearn.calibration import CalibratedClassifierCV

calibrated = {}
brier_platt     = {}
brier_isotonic  = {}

for name, model in models.items():
    cal_platt = CalibratedClassifierCV(model, cv='prefit', method='sigmoid')
    cal_platt.fit(X_val, y_val)

    cal_iso = CalibratedClassifierCV(model, cv='prefit', method='isotonic')
    cal_iso.fit(X_val, y_val)

    p_platt   = cal_platt.predict_proba(X_val)[:, 1]
    p_iso     = cal_iso.predict_proba(X_val)[:, 1]

    brier_platt[name]    = brier_score_loss(y_val, p_platt)
    brier_isotonic[name] = brier_score_loss(y_val, p_iso)
    calibrated[name]     = {'platt': cal_platt, 'isotonic': cal_iso}

results = pd.DataFrame({
    'Before':   brier_before,
    'Platt':    brier_platt,
    'Isotonic': brier_isotonic,
}).T
print(results.round(4).to_string())


In [ ]:
# ── Reliability curves after calibration (XGBoost) ───────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('XGBoost Reliability — Before vs After Calibration', fontsize=13)

xgb_raw   = models['XGBoost']
xgb_platt = calibrated['XGBoost']['platt']
xgb_iso   = calibrated['XGBoost']['isotonic']

for ax, (label, mdl) in zip(axes, [
    ('Uncalibrated', xgb_raw),
    ('Platt',        xgb_platt),
    ('Isotonic',     xgb_iso),
]):
    p = mdl.predict_proba(X_val)[:, 1]
    fp, mp = calibration_curve(y_val, p, n_bins=10, strategy='uniform')
    bs = brier_score_loss(y_val, p)
    ax.plot(mp, fp, 's-', label=f'{label} (Brier={bs:.4f})')
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set_title(label)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.legend(fontsize=8); ax.set_xlim(0,1); ax.set_ylim(0,1)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'reliability_diagrams.png'), dpi=150)
plt.show()


In [ ]:
# ── Save calibrated XGBoost (Platt) ──────────────────────────
xgb_calibrated = calibrated['XGBoost']['platt']
save_path = os.path.join(MODELS_DIR, 'xgb_calibrated.joblib')
joblib.dump(xgb_calibrated, save_path)
print(f"Saved calibrated XGBoost → {save_path}")


In [ ]:
# ── Threshold analysis on validation set ─────────────────────
proba_val = xgb_calibrated.predict_proba(X_val)[:, 1]

# F1-optimal threshold
precisions, recalls, thresholds_pr = precision_recall_curve(y_val, proba_val)
f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)
t_f1 = float(thresholds_pr[np.argmax(f1_scores)])

# Cost-optimal threshold: C_FN * FN + C_FP * FP
thresholds = np.linspace(0.01, 0.99, 200)
costs = []
for t in thresholds:
    preds = (proba_val >= t).astype(int)
    fp = int(np.sum((preds == 1) & (y_val == 0)))
    fn = int(np.sum((preds == 0) & (y_val == 1)))
    costs.append(C_FN * fn + C_FP * fp)
costs = np.array(costs)
t_cost = float(thresholds[np.argmin(costs)])
t_default = 0.5

print(f"t_default = {t_default:.2f}")
print(f"t_f1      = {t_f1:.3f}  (F1-optimal)")
print(f"t_cost    = {t_cost:.3f}  (cost-optimal, C_FN={C_FN}, C_FP={C_FP})")


In [ ]:
# ── Cost vs threshold curve ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, costs, lw=2, color='steelblue', label='Total cost')
for t_val, label, color in [
    (t_default, f't_default={t_default:.2f}', 'gray'),
    (t_f1,      f't_F1={t_f1:.2f}',          'green'),
    (t_cost,    f't_cost={t_cost:.2f}',       'crimson'),
]:
    idx = np.argmin(np.abs(thresholds - t_val))
    ax.axvline(t_val, color=color, linestyle='--', alpha=0.8)
    ax.scatter(thresholds[idx], costs[idx], color=color, zorder=5, s=80)
    ax.text(t_val + 0.01, costs[idx] + 5, label, color=color, fontsize=9)

ax.set_xlabel('Decision Threshold')
ax.set_ylabel(f'Cost  (C_FN={C_FN} × FN + C_FP={C_FP} × FP)')
ax.set_title('Cost vs Decision Threshold — Calibrated XGBoost (Validation Set)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'cost_threshold_curve.png'), dpi=150)
plt.show()


In [ ]:
# ── Sensitivity: vary cost ratio 2:1 to 10:1 ─────────────────
ratios  = np.arange(2, 11)
t_costs = []
for r in ratios:
    c = r * np.array([int(np.sum((proba_val >= t) & (y_val == 0))) for t in thresholds], dtype=float) +         np.array([int(np.sum((proba_val < t)  & (y_val == 1))) for t in thresholds], dtype=float)
    t_costs.append(thresholds[np.argmin(c)])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ratios, t_costs, 'o-', color='crimson', lw=2)
ax.set_xlabel('Cost ratio  C(FN) / C(FP)')
ax.set_ylabel('Optimal threshold')
ax.set_title('Optimal Threshold vs Cost Ratio')
ax.set_xticks(ratios)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'threshold_sensitivity.png'), dpi=150)
plt.show()
print(dict(zip(ratios.tolist(), [f'{t:.3f}' for t in t_costs])))


In [ ]:
# ── Save threshold values for downstream notebooks ────────────
import json
thresh_path = os.path.join(MODELS_DIR, 'thresholds.json')
with open(thresh_path, 'w') as f:
    json.dump({'t_default': t_default, 't_f1': t_f1, 't_cost': t_cost}, f, indent=2)
print(f"Thresholds saved → {thresh_path}")
